# 🏆 Flagship Lab (Enhanced): Fun & Fit Health Advisor Agent (GitHub Models)

Now with:
- ✅ Multiple tools
- ✅ Clear tool vs no-tool scenarios
- ✅ Debug prints (which tool was used)
- ✅ Clear explanation: LLM vs Agent

---

## 🔧 Setup

In [ ]:
from openai import OpenAI
import os, json

client = OpenAI(
    api_key=os.getenv("GITHUB_TOKEN"),
    base_url="https://models.inference.ai.azure.com"
)

model = "gpt-4.1"
print("Setup done")

## 🧠 Agent Personality

In [ ]:
system_prompt = """
You are a Fun & Fit Health Advisor Agent.

Rules:
- Friendly & motivating
- Short actionable advice
- If calculation needed → use tools
- Always personalize response
"""

## 🤔 Same Prompt with LLM (No Tools)

👉 YES — you can give same system prompt to LLM.
👉 But it **cannot execute tools**.


In [ ]:
response = client.chat.completions.create(
    model=model,
    messages=[
        {"role":"system","content":system_prompt},
        {"role":"user","content":"My weight is 85kg and height is 1.75m"}
    ]
)

print(response.choices[0].message.content)

🔍 Observation:
- It may GUESS BMI
- But it does NOT calculate using code

---

## 🛠️ Define Multiple Tools

In [ ]:
def calculate_bmi(weight, height):
    bmi = weight/(height**2)
    return f"BMI: {round(bmi,2)}"

def calorie_estimator(weight):
    return f"Estimated daily calories: {weight*30} kcal"

def workout_plan(goal):
    return f"Workout plan for {goal}: 30 mins cardio + strength training"


In [ ]:
tools = [
    {
        "type":"function",
        "function":{
            "name":"calculate_bmi",
            "description":"Calculate BMI",
            "parameters":{
                "type":"object",
                "properties":{
                    "weight":{"type":"number"},
                    "height":{"type":"number"}
                },
                "required":["weight","height"]
            }
        }
    },
    {
        "type":"function",
        "function":{
            "name":"calorie_estimator",
            "description":"Estimate calories",
            "parameters":{
                "type":"object",
                "properties":{
                    "weight":{"type":"number"}
                },
                "required":["weight"]
            }
        }
    },
    {
        "type":"function",
        "function":{
            "name":"workout_plan",
            "description":"Suggest workout",
            "parameters":{
                "type":"object",
                "properties":{
                    "goal":{"type":"string"}
                },
                "required":["goal"]
            }
        }
    }
]

## 🚫 Scenario 1: NO TOOL SHOULD BE USED

In [ ]:
response = client.chat.completions.create(
    model=model,
    messages=[
        {"role":"system","content":system_prompt},
        {"role":"user","content":"Give me motivation to stay fit"}
    ],
    tools=tools
)

print(response.choices[0].message)

## 🔧 Scenario 2: TOOL SHOULD BE USED

In [ ]:
response = client.chat.completions.create(
    model=model,
    messages=[
        {"role":"system","content":system_prompt},
        {"role":"user","content":"My weight is 90kg and height is 1.8m"}
    ],
    tools=tools
)

print(response.choices[0].message)

## 🔁 Agent Loop with Debug (SEE WHICH TOOL IS USED)

In [ ]:
messages = [
    {"role":"system","content":system_prompt},
    {"role":"user","content":"I weigh 85kg, height 1.75m, suggest calories and workout"}
]

while True:
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        tools=tools
    )

    msg = response.choices[0].message

    if not msg.tool_calls:
        print("\n✅ FINAL ANSWER:")
        print(msg.content)
        break

    tool_call = msg.tool_calls[0]
    name = tool_call.function.name
    args = json.loads(tool_call.function.arguments)

    print(f"\n🔧 TOOL CALLED: {name}")
    print("INPUT:", args)

    if name == "calculate_bmi":
        result = calculate_bmi(**args)
    elif name == "calorie_estimator":
        result = calorie_estimator(**args)
    elif name == "workout_plan":
        result = workout_plan(**args)

    print("OUTPUT:", result)

    messages.append(msg)
    messages.append({
        "role":"tool",
        "tool_call_id":tool_call.id,
        "content":result
    })

## 🧠 FINAL CONCEPT: LLM vs Agent

### LLM:
- Can follow instructions
- Can simulate reasoning
- ❌ Cannot ACT

### Agent:
- Uses LLM + Tools + Loop
- Can take ACTIONS
- Can do multi-step reasoning

👉 EXTRA THING = **ACTION + TOOL USAGE + LOOP**

---